In [1]:
import pandas as pd
import numpy as np 
from datetime import datetime

pd.set_option('display.max_columns', None)

In [2]:
FILE_PATH = r"C:\Users\feon4\OneDrive\المستندات\healthcare_dataset.csv"
df = pd.read_csv(FILE_PATH)

In [3]:
print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [4]:
df_original = df.copy()

df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

date_cols = ["date_of_adimssion", "discharge_date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df.columns.tolist()

['name',
 'age',
 'gender',
 'blood_type',
 'medical_condition',
 'date_of_admission',
 'doctor',
 'hospital',
 'insurance_provider',
 'billing_amount',
 'room_number',
 'admission_type',
 'discharge_date',
 'medication',
 'test_results']

In [5]:
def assess_accuracy(df):
    report = {}
    report["invalid_age_count"] = df[(df["age"] < 0) | (df["age"] > 120)].shape[0]
    report["invalid_billing_count"] = df[df["billing_amount"] < 0].shape[0]

    for col in date_cols:
        if col in df.columns:
            report[f"unparseable_{col}"] = df[col].isna().sum()
            
    valid_blood_types = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
    report["invalid_blood_type_count"] = df[~df["blood_type"].isin(valid_blood_types)].shape[0]
    return report

assess_accuracy(df)

{'invalid_age_count': 0,
 'invalid_billing_count': 108,
 'unparseable_discharge_date': 0,
 'invalid_blood_type_count': 0}

In [6]:
def assess_completeness(df):
    report = {}
    report["missing_per_column"] = df.isnull().sum().to_dict()

    total_cells = df.shape[0] * df.shape[1]
    total_missing = df.isnull().sum().sum()
    report["overall_missing_pct"] = round((total_missing / total_cells) * 100, 2)

    report["rows_missing_multi_cols"] = df[df.isnull().sum(axis=1) >= 2].shape[0]
    return report

assess_completeness(df)

{'missing_per_column': {'name': 0,
  'age': 0,
  'gender': 0,
  'blood_type': 0,
  'medical_condition': 0,
  'date_of_admission': 0,
  'doctor': 0,
  'hospital': 0,
  'insurance_provider': 0,
  'billing_amount': 0,
  'room_number': 0,
  'admission_type': 0,
  'discharge_date': 0,
  'medication': 0,
  'test_results': 0},
 'overall_missing_pct': 0.0,
 'rows_missing_multi_cols': 0}

In [7]:
def assess_consistency(df):
    report = {}
    if "name" in df.columns:
        report["inconsistent_name_casing"] = df[df["name"] != df["name"].str.title()].shape[0]

    if "gender" in df.columns:
        report["unique_gender_values"] = df["gender"].unique().tolist()

    if "admission_type" in df.columns:
        report["unique_admission_types"] = df["admission_type"].unique().tolist()

    if all(c in df.columns for c in date_cols):
        report["discharge_before_admission"] = df[df["discharge_date"] < df["date_of_admission"]].shape[0]
    return report 

assess_consistency(df)

{'inconsistent_name_casing': 55467,
 'unique_gender_values': ['Male', 'Female'],
 'unique_admission_types': ['Urgent', 'Emergency', 'Elective']}

In [8]:
relevant_columns = [
    "name", "age", "gender", "blood_type", "medical_condition",
    "date_of_admission", "doctor", "hospital", "insurance_provider",
    "billing_amount", "room_number", "admission_type", "discharge_date",
    "medication", "test_results"
]
def assess_relevance(df, relevant_columns):
    all_columns = df.columns.tolist()
    irrelevant_columns = [c for c in all_columns if c not in relevant_columns]
    return {"irrelevant_columns_found": irrelevant_columns, "relevant_columns_kept": relevant_columns}

def remove_irrelevant_columns(df, relevant_columns):
    return df[[c for c in relevant_columns if c in df.columns]]

assess_relevance(df, relevant_columns)

{'irrelevant_columns_found': [],
 'relevant_columns_kept': ['name',
  'age',
  'gender',
  'blood_type',
  'medical_condition',
  'date_of_admission',
  'doctor',
  'hospital',
  'insurance_provider',
  'billing_amount',
  'room_number',
  'admission_type',
  'discharge_date',
  'medication',
  'test_results']}

In [9]:
def assess_timeliness(df, reference_date=None):
    report = {}
    if reference_date is None:
        reference_date = pd.Timestamp(datetime.now())

    if "date_of_admission" in df.columns:
        report["min_admission_date"] = df["date_of_admission"].min()
        report["max_admission_date"] = df["date_of_admission"].max()
        cutoff = reference_date - pd.DateOffset(years=5)
        df["date_of_admission"] = pd.to_datetime(df["date_of_admission"])
        report["outdated_rows_count"] = df[df["date_of_admission"] < cutoff].shape[0]
    return report

assess_timeliness(df)

{'min_admission_date': '2019-05-08',
 'max_admission_date': '2024-05-07',
 'outdated_rows_count': 24881}

In [10]:
def assess_uniqueness(df):
    report = {}
    report["duplicate_rows_count"] = df.duplicated().sum()

    if all(c in df.columns for c in ["name", "date_of_admission"]):
        report["duplicate_name_admission_combo"] = df.duplicated(subset=["name", "date_of_admission"]).sum()
    return report

def remove_duplicate_rows(df):
    return df.drop_duplicates()

assess_uniqueness(df)

{'duplicate_rows_count': 534, 'duplicate_name_admission_combo': 5500}

In [11]:
def assess_validity(df):
    report = {}
    for col in date_cols:
        if col in df.columns:
            report[f"invalid_format_{col}"] = df[col].isna().sum()

    if "test_results" in df.columns:
        valid_results = ["Normal", "Abnormal", "Inconclusive"]
        report["invalid_test_results"] = df[~df["test_results"].isin(valid_results)].shape[0]

    invalid_mask = pd.Series(False, index=df.index)
    if "age" in df.columns:
        invalid_mask |= (df["age"] < 0) | (df["age"] > 120)
    if "billing_amount" in df.columns:
        invalid_mask |= (df["billing_amount"] < 0)
    report["rows_with_invalid_values"] = invalid_mask.sum()
    return report

assess_validity(df)

{'invalid_format_discharge_date': 0,
 'invalid_test_results': 0,
 'rows_with_invalid_values': 108}

In [12]:
before_report = {
    "accuracy": assess_accuracy(df),
    "completeness": assess_completeness(df),
    "consistency": assess_consistency(df),
    "relevance": assess_relevance(df, relevant_columns),
    "timeliness": assess_timeliness(df),
    "uniqueness": assess_uniqueness(df),
    "validity": assess_validity(df),
}
for dim, result in before_report.items():
    print(f"\n--- {dim.upper()} ---")
    print(result)


--- ACCURACY ---
{'invalid_age_count': 0, 'invalid_billing_count': 108, 'unparseable_discharge_date': 0, 'invalid_blood_type_count': 0}

--- COMPLETENESS ---
{'missing_per_column': {'name': 0, 'age': 0, 'gender': 0, 'blood_type': 0, 'medical_condition': 0, 'date_of_admission': 0, 'doctor': 0, 'hospital': 0, 'insurance_provider': 0, 'billing_amount': 0, 'room_number': 0, 'admission_type': 0, 'discharge_date': 0, 'medication': 0, 'test_results': 0}, 'overall_missing_pct': 0.0, 'rows_missing_multi_cols': 0}

--- CONSISTENCY ---
{'inconsistent_name_casing': 55467, 'unique_gender_values': ['Male', 'Female'], 'unique_admission_types': ['Urgent', 'Emergency', 'Elective']}

--- RELEVANCE ---
{'irrelevant_columns_found': [], 'relevant_columns_kept': ['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']}

--- TIMELINESS ---
{'

In [13]:
df_clean = df.copy()

critical_cols = ["name", "age", "medical_condition"]
df_clean = df_clean.dropna(subset=[c for c in critical_cols if c in df_clean.columns])

if "name" in df_clean.columns:
    df_clean["name"] = df_clean["name"].str.title()
for col in ["gender", "medical_condition", "admission_type", "test_results",
            "hospital", "insurance_provider", "medication", "doctor"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.strip().str.title()

df_clean = remove_duplicate_rows(df_clean)

if "age" in df_clean.columns:
    df_clean = df_clean[(df_clean["age"] >= 0) & (df_clean["age"] <= 120)]
if "billing_amount" in df_clean.columns:
    df_clean = df_clean[df_clean["billing_amount"] >= 0]

df_clean = remove_irrelevant_columns(df_clean, relevant_columns)

print("Shape after cleaning:", df_clean.shape)
df_clean.head()

Shape after cleaning: (54860, 15)


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [14]:
after_report = {
    "accuarcy": assess_accuracy(df_clean),
    "completeness": assess_completeness(df_clean),
    "consistency": assess_consistency(df_clean),
    "relevance": assess_relevance(df_clean, relevant_columns),
    "timeliness": assess_timeliness(df_clean),
    "uniqueness": assess_uniqueness(df_clean),
    "validity": assess_validity(df_clean),
}

for dim, result in after_report.items():
    print(f"\n--- {dim.upper()} ---")
    print(result)


--- ACCUARCY ---
{'invalid_age_count': 0, 'invalid_billing_count': 0, 'unparseable_discharge_date': 0, 'invalid_blood_type_count': 0}

--- COMPLETENESS ---
{'missing_per_column': {'name': 0, 'age': 0, 'gender': 0, 'blood_type': 0, 'medical_condition': 0, 'date_of_admission': 0, 'doctor': 0, 'hospital': 0, 'insurance_provider': 0, 'billing_amount': 0, 'room_number': 0, 'admission_type': 0, 'discharge_date': 0, 'medication': 0, 'test_results': 0}, 'overall_missing_pct': 0.0, 'rows_missing_multi_cols': 0}

--- CONSISTENCY ---
{'inconsistent_name_casing': 0, 'unique_gender_values': ['Male', 'Female'], 'unique_admission_types': ['Urgent', 'Emergency', 'Elective']}

--- RELEVANCE ---
{'irrelevant_columns_found': [], 'relevant_columns_kept': ['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']}

--- TIMELINESS ---
{'min_ad

In [15]:
def overall_quality_score(df):
    total_rows = df.shape[0]
    if total_rows == 0:
        return 0
    issues_mask = pd.Series(False, index=df.index)
    if "age" in df.columns:
        issues_mask |= (df["age"] < 0) | (df["age"] > 120)
    if "billing_amount" in df.columns:
        issues_mask |= (df["billing_amount"] < 0)

    issues_mask |= df.isnull().any(axis=1)
    issues_mask |=df.duplicated()

    clean_rows = total_rows - issues_mask.sum()
    return round((clean_rows / total_rows) * 100, 2)

score_before = overall_quality_score(df)
score_after = overall_quality_score(df_clean)

print(f"Rows before cleaning : {df.shape[0]}  | Quality Score:{score_before}%")
print(f"Rows after cleaning : {df_clean.shape[0]} | Quality Score: {score_after}%")

Rows before cleaning : 55500  | Quality Score:98.85%
Rows after cleaning : 54860 | Quality Score: 100.0%


In [16]:
summary_df = pd.DataFrame({
    "Stage": ["Before Cleaning", "After Cleaning"],
    "Row_Count": [df.shape[0], df_clean.shape[0]],
    "Quality_Score_%": [score_before, score_after],
})

summary_df.to_csv("quality_score_summary.csv", index=False)
df_clean.to_csv("healthcare_dataset_cleaned.csv", index=False)

summary_df

,Stage,Row_Count,Quality_Score_%
0,Before Cleaning,55500,98.85
1,After Cleaning,54860,100.00
